In [1]:
import os, json, pandas as pd, numpy as np
from tqdm import tqdm

# ── CONFIG — adjust these paths ───────────────────────────────────────────────
QTAIM_ROOT  = "/home/suba/Downloads/qtaim_generator/qtaim_generator/data/qm9/QTAIM/" 
TRAIN_PKL   = "/home/suba/Documents/GitHub/qtaim_embed_private/data_suba/train_qm9_qtaim_1205_labelled_corrected_my43k.pkl"  # local copy
TEST_PKL    = "/home/suba/Documents/GitHub/qtaim_embed_private/data_suba/test_qm9_qtaim_1205_labelled_corrected_my43k.pkl"   # local copy

# ── load and merge ────────────────────────────────────────────────────────────
train = pd.read_pickle(TRAIN_PKL)
test  = pd.read_pickle(TEST_PKL)
train["split"] = "train"
test["split"]  = "test"
df_local = pd.concat([train, test], ignore_index=True)
df_local["gdb_num"] = df_local["names"].str.extract(r"gdb_(\d+)\.xyz").astype(int)
print(f"Loaded {len(df_local)} rows")

# ── check how many qtaim.json folders exist ───────────────────────────────────
qtaim_folders = set(int(f) for f in os.listdir(QTAIM_ROOT)
                    if os.path.isdir(os.path.join(QTAIM_ROOT, f)) and f.isdigit())
print(f"QTAIM folders on disk: {len(qtaim_folders)}")
overlap = set(df_local["gdb_num"]) & qtaim_folders
print(f"Overlap with PKL     : {len(overlap)}")

# ── compare bonds column vs qtaim.json for every molecule we have ─────────────
match_count    = 0
mismatch_count = 0
missing_qtaim  = 0
results        = []

for _, row in tqdm(df_local.iterrows(), total=len(df_local)):
    gdb_num    = row["gdb_num"]
    qtaim_path = os.path.join(QTAIM_ROOT, str(gdb_num), "qtaim.json")

    if not os.path.exists(qtaim_path):
        missing_qtaim += 1
        continue

    with open(qtaim_path) as f:
        qtaim = json.load(f)

    # pairs from qtaim.json BCP keys
    qtaim_pairs = set()
    for k in qtaim:
        if '_' in str(k):
            i,j = int(k.split('_')[0]), int(k.split('_')[1])
            qtaim_pairs.add((min(i,j), max(i,j)))

    # pairs from PKL bonds column
    raw = row["bonds"]
    if isinstance(raw, list) and len(raw) == 1:
        raw = raw[0]
    pkl_pairs = {(min(i,j), max(i,j)) for i,j in raw if i != j}

    if qtaim_pairs == pkl_pairs:
        match_count += 1
    else:
        mismatch_count += 1
        results.append({
            "gdb_num"  : gdb_num,
            "n_qtaim"  : len(qtaim_pairs),
            "n_pkl"    : len(pkl_pairs),
            "n_overlap": len(qtaim_pairs & pkl_pairs),
            "missing_from_pkl" : sorted(qtaim_pairs - pkl_pairs)[:3],
            "phantom_in_pkl"   : sorted(pkl_pairs - qtaim_pairs)[:3],
        })

print(f"\n{'='*50}")
print(f"Total checked    : {match_count + mismatch_count}")
print(f"Missing qtaim    : {missing_qtaim}")
print(f"PKL matches qtaim: {match_count}")
print(f"PKL mismatches   : {mismatch_count}")
print(f"Mismatch rate    : {mismatch_count/(match_count+mismatch_count)*100:.1f}%")

if results:
    print(f"\nFirst 15 mismatches:")
    print(f"  {'gdb':>8}  {'n_qtaim':>7}  {'n_pkl':>5}  {'overlap':>7}  "
          f"{'missing_from_pkl(sample)':>30}  {'phantom_in_pkl(sample)'}")
    print("  " + "-"*90)
    for r in results[:15]:
        print(f"  {r['gdb_num']:>8}  {r['n_qtaim']:>7}  {r['n_pkl']:>5}  "
              f"{r['n_overlap']:>7}  {str(r['missing_from_pkl']):>30}  "
              f"{r['phantom_in_pkl']}")

    # check if the overlap is always 0 (completely wrong molecule)
    # or partial (index shift)
    zero_overlap  = sum(1 for r in results if r['n_overlap'] == 0)
    some_overlap  = sum(1 for r in results if 0 < r['n_overlap'] < r['n_qtaim'])
    full_match    = sum(1 for r in results if r['n_overlap'] == r['n_qtaim'])
    print(f"\nOf the {mismatch_count} mismatches:")
    print(f"  zero overlap (completely wrong mol) : {zero_overlap}")
    print(f"  partial overlap (index shift?)      : {some_overlap}")
    print(f"  full qtaim covered (extra phantoms) : {full_match}")

Loaded 43476 rows
QTAIM folders on disk: 133885
Overlap with PKL     : 43475


100%|██████████| 43476/43476 [00:21<00:00, 2022.13it/s]


Total checked    : 43475
Missing qtaim    : 1
PKL matches qtaim: 0
PKL mismatches   : 43475
Mismatch rate    : 100.0%

First 15 mismatches:
       gdb  n_qtaim  n_pkl  overlap        missing_from_pkl(sample)  phantom_in_pkl(sample)
  ------------------------------------------------------------------------------------------
     78284       12     13        0        [(0, 1), (0, 7), (1, 2)]  [(1, 3), (1, 11), (3, 5)]
     21533       22      7        4      [(0, 1), (0, 10), (0, 11)]  [(0, 2), (2, 4), (4, 10)]
     95862       14     13        4       [(0, 9), (1, 2), (1, 10)]  [(0, 11), (1, 3), (3, 12)]
     23567       19     13        5       [(0, 10), (1, 2), (1, 3)]  [(1, 5), (5, 8), (5, 11)]
     68168       16     13        5        [(0, 1), (1, 2), (1, 9)]  [(1, 8), (1, 11), (3, 5)]
     81594       24     11        2      [(0, 1), (0, 10), (0, 11)]  [(0, 2), (2, 5), (2, 10)]
     65910       25     12        3       [(0, 1), (0, 9), (0, 10)]  [(1, 5), (1, 11), (3, 5)]
     968

In [2]:
import os, json
from tqdm import tqdm
from collections import Counter

QTAIM_ROOT = "/home/suba/Downloads/qtaim_generator/qtaim_generator/data/qm9/QTAIM"

shift_counter   = Counter()
pattern_samples = []

for _, row in tqdm(df_local.iterrows(), total=len(df_local)):
    gdb_num    = row["gdb_num"]
    qtaim_path = os.path.join(QTAIM_ROOT, str(gdb_num), "qtaim.json")
    if not os.path.exists(qtaim_path):
        continue

    with open(qtaim_path) as f:
        qtaim = json.load(f)

    # qtaim.json BCP pairs
    qtaim_pairs = set()
    for k in qtaim:
        if '_' in str(k):
            i,j = int(k.split('_')[0]), int(k.split('_')[1])
            qtaim_pairs.add((min(i,j), max(i,j)))

    # PKL bonds column pairs
    raw = row["bonds"]
    if isinstance(raw, list) and len(raw) == 1:
        raw = raw[0]
    pkl_pairs = {(min(i,j), max(i,j)) for i,j in raw if i != j}

    # try every possible constant shift: pkl_pair - shift = qtaim_pair?
    best_shift  = None
    best_overlap = 0
    for shift in range(-5, 20):
        shifted = {(min(i+shift, j+shift), max(i+shift, j+shift))
                   for i,j in pkl_pairs}
        overlap = len(shifted & qtaim_pairs)
        if overlap > best_overlap:
            best_overlap = overlap
            best_shift   = shift

    total = len(qtaim_pairs)
    shift_counter[best_shift] += 1

    if len(pattern_samples) < 20:
        pattern_samples.append({
            "gdb"        : gdb_num,
            "best_shift" : best_shift,
            "overlap_after_shift" : best_overlap,
            "total_qtaim": total,
            "perfect"    : best_overlap == total,
        })

print("=== Shift distribution (pkl index + shift = qtaim index) ===")
for shift, count in sorted(shift_counter.items(), key=lambda x: -x[1]):
    pct = count / sum(shift_counter.values()) * 100
    print(f"  shift={shift:+3d}  count={count:6d}  ({pct:.1f}%)")

print(f"\nSample molecules:")
print(f"  {'gdb':>8}  {'best_shift':>10}  {'overlap/qtaim':>14}  perfect?")
print("  " + "-"*50)
for s in pattern_samples:
    print(f"  {s['gdb']:>8}  {s['best_shift']:>10}  "
          f"{s['overlap_after_shift']:>6}/{s['total_qtaim']:<6}  "
          f"{'✓' if s['perfect'] else '✗'}")

print(f"\nTotal molecules: {sum(shift_counter.values())}")

100%|██████████| 43476/43476 [00:16<00:00, 2695.08it/s]


=== Shift distribution (pkl index + shift = qtaim index) ===
  shift= +0  count= 11108  (25.6%)
  shift= -1  count= 10899  (25.1%)
  shift= -2  count=  6791  (15.6%)
  shift= -3  count=  4707  (10.8%)
  shift= +1  count=  3237  (7.4%)
  shift= -4  count=  2849  (6.6%)
  shift= -5  count=  1770  (4.1%)
  shift= +2  count=  1311  (3.0%)
  shift= +3  count=   529  (1.2%)
  shift= +4  count=   197  (0.5%)
  shift= +5  count=    55  (0.1%)
  shift= +6  count=    16  (0.0%)


TypeError: unsupported format string passed to NoneType.__format__

In [3]:
import os

QTAIM_GEN_ROOT = "/home/suba/Downloads/qtaim_generator/qtaim_generator"

# find all python files
py_files = []
for root, dirs, files in os.walk(QTAIM_GEN_ROOT):
    for f in files:
        if f.endswith(".py"):
            py_files.append(os.path.join(root, f))

print(f"Total .py files: {len(py_files)}")
print()

# search for where 'bonds' column is written
print("=== Files that mention 'bonds' ===")
for fp in py_files:
    try:
        txt = open(fp).read()
        if "bonds" in txt.lower():
            lines = [(i+1, l.strip()) for i,l in enumerate(txt.splitlines())
                     if "bonds" in l.lower() and not l.strip().startswith("#")]
            if lines:
                print(f"\n  {fp}")
                for lineno, line in lines[:15]:
                    print(f"    {lineno:4d}: {line}")
    except:
        pass

print()
# search for where 'bond_paths' or 'connected_bond_paths' is used
print("=== Files that mention 'connected_bond_paths' or 'bond_path' ===")
for fp in py_files:
    try:
        txt = open(fp).read()
        if "connected_bond_paths" in txt or "bond_path" in txt.lower():
            lines = [(i+1, l.strip()) for i,l in enumerate(txt.splitlines())
                     if "bond_path" in l.lower() or "connected_bond_paths" in l]
            if lines:
                print(f"\n  {fp}")
                for lineno, line in lines[:15]:
                    print(f"    {lineno:4d}: {line}")
    except:
        pass

Total .py files: 0

=== Files that mention 'bonds' ===

=== Files that mention 'connected_bond_paths' or 'bond_path' ===


In [4]:
# simulate what mol_wrappers_from_df does for gdb_067265
import json

SAMPLE_GDB = 67265
row = df_local[df_local["gdb_num"] == SAMPLE_GDB].iloc[0]

print("=== raw row[bonds] ===")
raw = row["bonds"]
print(f"type={type(raw)}  len={len(raw)}")
print(f"value={raw}")

print("\n=== line 2: bonds = {tuple(sorted(b)): None for b in bonds} ===")
print("iterating over raw gives:")
for i, b in enumerate(raw):
    print(f"  item {i}: type={type(b)}  value={b}")
    result = tuple(sorted(b))
    print(f"  tuple(sorted(b)) = {result[:5]}...  len={len(result)}")

wrong_bonds = {tuple(sorted(b)): None for b in raw}
print(f"\nresulting bonds dict has {len(wrong_bonds)} keys:")
for k in wrong_bonds:
    print(f"  {k[:5]}...  len={len(k)}")

print("\n=== line 3: if len(row[bond_key]) == 1: bonds = row[bond_key][0] ===")
print(f"len(raw) = {len(raw)}")
if len(raw) == 1:
    bonds_corrected = raw[0]
    print(f"unwrapped: {bonds_corrected}")
    correct_dict = {tuple(sorted(b)): None for b in bonds_corrected if b[0]!=b[1]}
    print(f"\ncorrected bonds dict ({len(correct_dict)} pairs):")
    for k in sorted(correct_dict):
        print(f"  {k}")

print("\n=== what get_bond_features sees ===")
print("get_bond_features is called BEFORE line 3 unwrap,")
print("using bond_key='bonds' which still has the nested structure.")
print("So it reads extra_feat_bond_indices_qtaim and builds features")
print("using whatever pairs are in that column — independently of bonds.")
print()

# show extra_feat_bond_indices_qtaim
qtaim_idx = row["extra_feat_bond_indices_qtaim"]
print(f"extra_feat_bond_indices_qtaim:")
print(f"  type={type(qtaim_idx)}")
print(f"  value={qtaim_idx}")

# load qtaim.json to compare
qtaim_path = f"/home/suba/Downloads/qtaim_generator/qtaim_generator/data/qm9/QTAIM/{SAMPLE_GDB}/qtaim.json"
with open(qtaim_path) as f:
    qtaim = json.load(f)
qtaim_pairs = sorted((min(int(k.split('_')[0]),int(k.split('_')[1])),
                       max(int(k.split('_')[0]),int(k.split('_')[1])))
                      for k in qtaim if '_' in str(k))
print(f"\nqtaim.json BCP pairs: {qtaim_pairs}")
print(f"\nDo qtaim.json pairs match extra_feat_bond_indices_qtaim?")
if isinstance(qtaim_idx, list):
    idx_pairs = sorted((min(i,j),max(i,j)) for i,j in qtaim_idx if i!=j)
    print(f"  extra_feat pairs: {idx_pairs}")
    print(f"  match: {idx_pairs == qtaim_pairs}")

=== raw row[bonds] ===
type=<class 'list'>  len=1
value=[[(5, 14), (5, 15), (5, 5), (5, 7), (5, 6), (5, 13), (5, 16), (3, 5), (3, 6), (5, 8), (6, 8), (6, 12), (1, 8), (1, 11)]]

=== line 2: bonds = {tuple(sorted(b)): None for b in bonds} ===
iterating over raw gives:
  item 0: type=<class 'list'>  value=[(5, 14), (5, 15), (5, 5), (5, 7), (5, 6), (5, 13), (5, 16), (3, 5), (3, 6), (5, 8), (6, 8), (6, 12), (1, 8), (1, 11)]
  tuple(sorted(b)) = ((1, 8), (1, 11), (3, 5), (3, 6), (5, 5))...  len=14

resulting bonds dict has 1 keys:
  ((1, 8), (1, 11), (3, 5), (3, 6), (5, 5))...  len=14

=== line 3: if len(row[bond_key]) == 1: bonds = row[bond_key][0] ===
len(raw) = 1
unwrapped: [(5, 14), (5, 15), (5, 5), (5, 7), (5, 6), (5, 13), (5, 16), (3, 5), (3, 6), (5, 8), (6, 8), (6, 12), (1, 8), (1, 11)]

corrected bonds dict (13 pairs):
  (1, 8)
  (1, 11)
  (3, 5)
  (3, 6)
  (5, 6)
  (5, 7)
  (5, 8)
  (5, 13)
  (5, 14)
  (5, 15)
  (5, 16)
  (6, 8)
  (6, 12)

=== what get_bond_features sees ===
get_bo

In [6]:
import os, json
from tqdm import tqdm

QTAIM_ROOT  = "/home/suba/Downloads/qtaim_generator/qtaim_generator/data/qm9/QTAIM"
SAMPLE_GDB  = 67265

# the pairs in the PKL for gdb_067265
row      = df_local[df_local["gdb_num"] == SAMPLE_GDB].iloc[0]
raw      = row["extra_feat_bond_indices_qtaim"]
pkl_pairs = frozenset((min(i,j), max(i,j)) for i,j in raw if i!=j)
print(f"PKL bond pairs for gdb_{SAMPLE_GDB}: {sorted(pkl_pairs)}")
print(f"n = {len(pkl_pairs)}")
print()

# search all qtaim.json folders for a match
print("Searching for which molecule's qtaim.json matches these pairs...")
found = []
folders = sorted(int(f) for f in os.listdir(QTAIM_ROOT)
                 if os.path.isdir(os.path.join(QTAIM_ROOT,f)) and f.isdigit())

for gdb in tqdm(folders):
    qpath = os.path.join(QTAIM_ROOT, str(gdb), "qtaim.json")
    if not os.path.exists(qpath):
        continue
    with open(qpath) as f:
        qtaim = json.load(f)
    q_pairs = frozenset((min(int(k.split('_')[0]),int(k.split('_')[1])),
                          max(int(k.split('_')[0]),int(k.split('_')[1])))
                         for k in qtaim if '_' in str(k))
    if q_pairs == pkl_pairs:
        found.append(gdb)

print(f"\nMatching qtaim.json found for: {found}")

# also check: do the PKL bond feature arrays match the qtaim.json of the found molecule?
if found:
    for match_gdb in found:
        print(f"\n=== gdb_{match_gdb} qtaim.json BCP pairs ===")
        qpath = os.path.join(QTAIM_ROOT, str(match_gdb), "qtaim.json")
        with open(qpath) as f:
            qtaim = json.load(f)
        bcp_keys = sorted([k for k in qtaim if '_' in str(k)],
                           key=lambda x: qtaim[x]['cp_num'])
        for k in bcp_keys:
            i,j = k.split('_')
            print(f"  ({i},{j})  e_density={qtaim[k].get('e_density',0):.4f}")

        # compare with PKL extra_feat_bond_e_density for gdb_067265
        print(f"\nPKL extra_feat_bond_e_density for gdb_{SAMPLE_GDB}:")
        raw_feat = row["extra_feat_bond_e_density"]
        if isinstance(raw_feat, list) and len(raw_feat)==1:
            raw_feat = raw_feat[0]
        print(f"  {[round(float(v),4) for v in raw_feat]}")

PKL bond pairs for gdb_67265: [(1, 8), (1, 11), (3, 5), (3, 6), (5, 6), (5, 7), (5, 8), (5, 13), (5, 14), (5, 15), (5, 16), (6, 8), (6, 12)]
n = 13

Searching for which molecule's qtaim.json matches these pairs...


100%|██████████| 133885/133885 [00:47<00:00, 2838.63it/s]


Matching qtaim.json found for: []


# Try to build graph from bond.json and qtaim.json files instead of using PKL data from qtaim_embed

In [8]:
import json, numpy as np, os

QTAIM_ROOT  = "/home/suba/Downloads/qtaim_generator/qtaim_generator/data/qm9/QTAIM"
SAMPLE_GDB  = 67265

with open(os.path.join(QTAIM_ROOT, str(SAMPLE_GDB), "qtaim.json")) as f: qtaim = json.load(f)
with open(os.path.join(QTAIM_ROOT, str(SAMPLE_GDB), "bond.json"))  as f: bond  = json.load(f)

# ── Step 1: atoms ─────────────────────────────────────────────────────────────
atom_keys = sorted([k for k in qtaim if '_' not in str(k)], key=lambda x: int(x))
n_atoms   = len(atom_keys)
atoms     = {int(k): {"element": qtaim[k]["element"],
                       "number":  qtaim[k]["number"],
                       "xyz":     qtaim[k]["pos_ang"]}
             for k in atom_keys}

print(f"=== Atoms from qtaim.json ({n_atoms}) ===")
for idx, v in atoms.items():
    print(f"  [{idx:2d}]  Multiwfn#{v['number']:2d}  {v['element']:2s}  {[round(x,4) for x in v['xyz']]}")

# ── Step 2: bond topology from qtaim.json BCP keys ───────────────────────────
bcp_keys = sorted([k for k in qtaim if '_' in str(k)],
                   key=lambda x: qtaim[x]['cp_num'])
qtaim_bonds = []
print(f"\n=== Bond topology from qtaim.json BCPs ({len(bcp_keys)}) ===")
for k in bcp_keys:
    i, j   = int(k.split('_')[0]), int(k.split('_')[1])
    sp_i   = atoms[i]['element']
    sp_j   = atoms[j]['element']
    lagr   = qtaim[k].get('Lagrangian_K', 0.0)
    qtaim_bonds.append((min(i,j), max(i,j)))
    print(f"  ({i:2d},{j:2d})  {sp_i}-{sp_j}  Lagrangian_K={lagr:.4f}")

# ── Step 3: bond topology from bond.json ibsi > 0.3 ──────────────────────────
bond_bonds = []
print(f"\n=== Bond topology from bond.json ibsi>0.3 ===")
for key, val in sorted(bond['ibsi'].items(), key=lambda x: -x[1]):
    if val < 0.3: break
    parts = key.split('_to_')
    i     = int(parts[0].split('_')[0]) - 1
    j     = int(parts[1].split('_')[0]) - 1
    sp_i  = parts[0].split('_')[1]
    sp_j  = parts[1].split('_')[1]
    bond_bonds.append((min(i,j), max(i,j)))
    print(f"  ({i:2d},{j:2d})  {sp_i}-{sp_j}  ibsi={val:.5f}")

print(f"\nqtaim.json bonds == bond.json bonds? {set(qtaim_bonds)==set(bond_bonds)}")

# ── Step 4: bond features from qtaim.json ────────────────────────────────────
BOND_FEAT_KEYS = [
    'Lagrangian_K','Hamiltonian_K','e_density','lap_e_density',
    'e_loc_func','ave_loc_ion_E','delta_g_promolecular','delta_g_hirsh',
    'esp_nuc','esp_e','esp_total','grad_norm','lap_norm',
    'eig_hess','det_hessian','ellip_e_dens','eta','energy_density','lol'
]
ATOM_FEAT_KEYS = BOND_FEAT_KEYS

bond_feat_dict = {}
for k in bcp_keys:
    i, j  = int(k.split('_')[0]), int(k.split('_')[1])
    pair  = (min(i,j), max(i,j))
    bond_feat_dict[pair] = {fk: qtaim[k].get(fk, 0.0) for fk in BOND_FEAT_KEYS}

atom_feat_dict = {}
for k in atom_keys:
    idx = int(k)
    atom_feat_dict[idx] = {fk: qtaim[k].get(fk, 0.0) for fk in ATOM_FEAT_KEYS}

print(f"\n=== Atom features (Lagrangian_K only) ===")
for idx, feat in atom_feat_dict.items():
    print(f"  [{idx:2d}] {atoms[idx]['element']:2s}  Lagr={feat['Lagrangian_K']:.4f}")

# ── Step 5: build graph edges (same as grapher.py) ────────────────────────────
bond_list   = sorted(bond_feat_dict.keys())
num_bonds   = len(bond_list)
num_atoms_g = n_atoms

a2b, b2a = [], []
for b_idx, (i,j) in enumerate(bond_list):
    a2b.extend([[i, b_idx], [j, b_idx]])
    b2a.extend([[b_idx, i], [b_idx, j]])

atoms_in_a2b = {p[0] for p in a2b}
disconnected = sorted(i for i in range(num_atoms_g) if i not in atoms_in_a2b)

print(f"\n=== Final graph from local qtaim.json ===")
print(f"  num_atoms        = {num_atoms_g}")
print(f"  num_bonds        = {num_bonds}")
print(f"  disconnected     = {disconnected}  ({'NONE ✓' if not disconnected else 'PROBLEM'})")

print(f"\n=== Bond nodes ===")
for b_idx, (i,j) in enumerate(bond_list):
    sp_i = atoms[i]['element']
    sp_j = atoms[j]['element']
    lagr = bond_feat_dict[(i,j)]['Lagrangian_K']
    print(f"  b{b_idx:2d}  ({i:2d},{j:2d})  {sp_i}-{sp_j}  Lagr={lagr:.4f}")

# ── Step 6: compare local vs PKL vs molecule_graph ───────────────────────────
row_pkl  = df_local[df_local["gdb_num"] == SAMPLE_GDB].iloc[0]
mg       = row_pkl["molecule_graph"]
mg_edges = {(min(u,v), max(u,v)) for u,v,_ in mg.graph.edges(data=True)}

local_set = set(bond_list)
pkl_raw   = row_pkl["bonds"]
if isinstance(pkl_raw, list) and len(pkl_raw)==1: pkl_raw = pkl_raw[0]
pkl_set   = {(min(i,j), max(i,j)) for i,j in pkl_raw if i!=j}

all_pairs = sorted(local_set | pkl_set | mg_edges)

print(f"\n=== Pair-by-pair comparison ===")
print(f"  {'pair':>10}  {'mg_graph':>8}  {'local_qtaim':>11}  {'PKL_bonds':>9}")
print("  " + "-"*45)
for p in all_pairs:
    m = "✓" if p in mg_edges  else "✗"
    l = "✓" if p in local_set else "✗"
    k = "✓" if p in pkl_set   else "✗"
    print(f"  ({p[0]:2d},{p[1]:2d})    {m:>8}  {l:>11}  {k:>9}")

print(f"\n  molecule_graph : {len(mg_edges)} bonds")
print(f"  local qtaim    : {len(local_set)} bonds")
print(f"  PKL bonds      : {len(pkl_set)} bonds")
print(f"\n  local == mg_graph? {local_set == mg_edges}")
print(f"  PKL   == mg_graph? {pkl_set   == mg_edges}")
print(f"\n  Missing from local : {sorted(mg_edges - local_set)}")
print(f"  Phantoms in local  : {sorted(local_set - mg_edges)}")

=== Atoms from qtaim.json (20) ===


ValueError: Unknown format code 'd' for object of type 'str'

In [9]:
import json, os

QTAIM_ROOT = "/home/suba/Downloads/qtaim_generator/qtaim_generator/data/qm9/QTAIM"
SAMPLE_GDB = 67265

with open(os.path.join(QTAIM_ROOT, str(SAMPLE_GDB), "qtaim.json")) as f:
    qtaim = json.load(f)

# bonds from qtaim.json
local_bonds = set()
for k in qtaim:
    if '_' in str(k):
        i, j = int(k.split('_')[0]), int(k.split('_')[1])
        local_bonds.add((min(i,j), max(i,j)))

# molecule_graph bonds
row_pkl  = df_local[df_local["gdb_num"] == SAMPLE_GDB].iloc[0]
mg       = row_pkl["molecule_graph"]
mg_edges = {(min(u,v), max(u,v)) for u,v,_ in mg.graph.edges(data=True)}

# PKL bonds
pkl_raw = row_pkl["bonds"]
if isinstance(pkl_raw, list) and len(pkl_raw)==1: pkl_raw = pkl_raw[0]
pkl_bonds = {(min(i,j), max(i,j)) for i,j in pkl_raw if i!=j}

print(f"molecule_graph : {len(mg_edges)} bonds  {sorted(mg_edges)}")
print(f"local qtaim    : {len(local_bonds)} bonds  {sorted(local_bonds)}")
print(f"PKL bonds      : {len(pkl_bonds)} bonds  {sorted(pkl_bonds)}")
print()
print(f"local == mg_graph? {local_bonds == mg_edges}")
print(f"PKL   == mg_graph? {pkl_bonds   == mg_edges}")

molecule_graph : 20 bonds  [(0, 1), (0, 9), (0, 10), (0, 11), (1, 2), (1, 4), (1, 8), (2, 3), (2, 6), (2, 12), (3, 4), (4, 5), (4, 13), (5, 6), (5, 8), (5, 14), (6, 7), (6, 15), (7, 8), (8, 16)]
local qtaim    : 20 bonds  [(0, 1), (0, 9), (0, 10), (0, 11), (1, 2), (2, 3), (2, 12), (2, 13), (3, 4), (3, 8), (3, 14), (4, 5), (4, 15), (4, 16), (5, 6), (6, 7), (6, 8), (7, 17), (8, 18), (8, 19)]
PKL bonds      : 13 bonds  [(1, 8), (1, 11), (3, 5), (3, 6), (5, 6), (5, 7), (5, 8), (5, 13), (5, 14), (5, 15), (5, 16), (6, 8), (6, 12)]

local == mg_graph? False
PKL   == mg_graph? False


In [11]:
import json, os, numpy as np

QTAIM_ROOT = "/home/suba/Downloads/qtaim_generator/qtaim_generator/data/qm9/QTAIM"

SAMPLE_GDB = 67265
row_pkl = df_local[df_local["gdb_num"] == SAMPLE_GDB].iloc[0]

pkl_id   = row_pkl["ids"]
pkl_name = row_pkl["names"]
print(f"PKL names={pkl_name}  ids={pkl_id}  gdb_num={SAMPLE_GDB}")

# try the ids field as folder name
id_path  = os.path.join(QTAIM_ROOT, str(pkl_id), "qtaim.json")
gdb_path = os.path.join(QTAIM_ROOT, str(SAMPLE_GDB), "qtaim.json")

print(f"\nDoes folder {pkl_id} exist?   {os.path.exists(id_path)}")
print(f"Does folder {SAMPLE_GDB} exist? {os.path.exists(gdb_path)}")

# load whichever exists and check coords
for label, path in [(f"ids={pkl_id}", id_path), (f"gdb={SAMPLE_GDB}", gdb_path)]:
    if not os.path.exists(path):
        print(f"\n{label}: folder not found")
        continue

    with open(path) as f:
        qtaim = json.load(f)

    atom_keys = sorted([k for k in qtaim if '_' not in str(k)], key=lambda x: int(x))

    # check coords against PKL molecule_graph
    mg      = row_pkl["molecule_graph"]
    mol     = mg.molecule
    n_match = 0
    for k in atom_keys:
        idx = int(k)
        if idx >= len(mol.sites): break
        q_xyz   = np.array(qtaim[k]["pos_ang"])
        pmg_xyz = np.array(mol.sites[idx].coords)
        diff    = np.max(np.abs(q_xyz - pmg_xyz))
        if diff < 0.05: n_match += 1

    bcp_pairs = set()
    for k in qtaim:
        if '_' in str(k):
            i,j = int(k.split('_')[0]), int(k.split('_')[1])
            bcp_pairs.add((min(i,j), max(i,j)))

    mg_edges = {(min(u,v), max(u,v)) for u,v,_ in mg.graph.edges(data=True)}

    print(f"\n{label} (folder {os.path.dirname(path).split('/')[-1]}):")
    print(f"  coord matches : {n_match}/{len(atom_keys)}")
    print(f"  bcp pairs     : {len(bcp_pairs)}")
    print(f"  mg_edges      : {len(mg_edges)}")
    print(f"  bcp == mg?    : {bcp_pairs == mg_edges}")
    print(f"  bcp pairs     : {sorted(bcp_pairs)}")
    print(f"  mg edges      : {sorted(mg_edges)}")

# now check a few more rows to confirm ids is the right key
print(f"\n=== First 10 rows: names vs ids vs gdb_num ===")
for _, r in df_local.head(10).iterrows():
    gdb = r["gdb_num"]
    rid = r["ids"]
    has_id_folder  = os.path.exists(os.path.join(QTAIM_ROOT, str(rid)))
    has_gdb_folder = os.path.exists(os.path.join(QTAIM_ROOT, str(gdb)))
    print(f"  names={r['names']:20s}  ids={rid:6d}  gdb={gdb:6d}  "
          f"folder_ids={'✓' if has_id_folder else '✗'}  "
          f"folder_gdb={'✓' if has_gdb_folder else '✗'}")

PKL names=gdb_67265.xyz  ids=8680  gdb_num=67265

Does folder 8680 exist?   True
Does folder 67265 exist? True

ids=8680 (folder 8680):
  coord matches : 17/17
  bcp pairs     : 20
  mg_edges      : 20
  bcp == mg?    : True
  bcp pairs     : [(0, 1), (0, 9), (0, 10), (0, 11), (1, 2), (1, 4), (1, 8), (2, 3), (2, 6), (2, 12), (3, 4), (4, 5), (4, 13), (5, 6), (5, 8), (5, 14), (6, 7), (6, 15), (7, 8), (8, 16)]
  mg edges      : [(0, 1), (0, 9), (0, 10), (0, 11), (1, 2), (1, 4), (1, 8), (2, 3), (2, 6), (2, 12), (3, 4), (4, 5), (4, 13), (5, 6), (5, 8), (5, 14), (6, 7), (6, 15), (7, 8), (8, 16)]

gdb=67265 (folder 67265):
  coord matches : 0/20
  bcp pairs     : 20
  mg_edges      : 20
  bcp == mg?    : False
  bcp pairs     : [(0, 1), (0, 9), (0, 10), (0, 11), (1, 2), (2, 3), (2, 12), (2, 13), (3, 4), (3, 8), (3, 14), (4, 5), (4, 15), (4, 16), (5, 6), (6, 7), (6, 8), (7, 17), (8, 18), (8, 19)]
  mg edges      : [(0, 1), (0, 9), (0, 10), (0, 11), (1, 2), (1, 4), (1, 8), (2, 3), (2, 6), (2, 1

In [12]:
import os, json, sys, numpy as np, pandas as pd

QTAIM_ROOT = "/home/suba/Downloads/qtaim_generator/qtaim_generator/data/qm9/QTAIM"
SAMPLE_GDB = 67265

row     = df_local[df_local["gdb_num"] == SAMPLE_GDB].iloc[0]
mol_id  = int(row["ids"])
print(f"names={row['names']}  ids={mol_id}  gdb_num={SAMPLE_GDB}")

# ── load correct qtaim.json using ids ─────────────────────────────────────────
qtaim_path = os.path.join(QTAIM_ROOT, str(mol_id), "qtaim.json")
bond_path  = os.path.join(QTAIM_ROOT, str(mol_id), "bond.json")

with open(qtaim_path) as f: qtaim = json.load(f)
with open(bond_path)  as f: bond  = json.load(f)

# ── rebuild bonds column correctly ───────────────────────────────────────────
atom_keys = sorted([k for k in qtaim if '_' not in str(k)], key=lambda x: int(x))
bcp_keys  = sorted([k for k in qtaim if '_'     in str(k)],
                    key=lambda x: qtaim[x]['cp_num'])

correct_bonds = []
for k in bcp_keys:
    i, j = int(k.split('_')[0]), int(k.split('_')[1])
    correct_bonds.append((i, j))

print(f"\nCorrect bonds from ids={mol_id} qtaim.json ({len(correct_bonds)}):")
for b in correct_bonds:
    print(f"  {b}")

# ── build corrected row with proper bonds and bond features ───────────────────
BOND_FEAT_KEYS = [
    "extra_feat_bond_Lagrangian_K", "extra_feat_bond_Hamiltonian_K",
    "extra_feat_bond_e_density",    "extra_feat_bond_lap_e_density",
    "extra_feat_bond_e_loc_func",   "extra_feat_bond_ave_loc_ion_E",
    "extra_feat_bond_delta_g_promolecular", "extra_feat_bond_delta_g_hirsh",
    "extra_feat_bond_esp_nuc",      "extra_feat_bond_esp_e",
    "extra_feat_bond_esp_total",    "extra_feat_bond_grad_norm",
    "extra_feat_bond_lap_norm",     "extra_feat_bond_eig_hess",
    "extra_feat_bond_det_hessian",  "extra_feat_bond_ellip_e_dens",
    "extra_feat_bond_eta",          "extra_feat_bond_energy_density",
    "extra_feat_bond_lol",
]
QTAIM_KEY_MAP = {  # PKL col → qtaim.json field
    "extra_feat_bond_Lagrangian_K":          "Lagrangian_K",
    "extra_feat_bond_Hamiltonian_K":         "Hamiltonian_K",
    "extra_feat_bond_e_density":             "e_density",
    "extra_feat_bond_lap_e_density":         "lap_e_density",
    "extra_feat_bond_e_loc_func":            "e_loc_func",
    "extra_feat_bond_ave_loc_ion_E":         "ave_loc_ion_E",
    "extra_feat_bond_delta_g_promolecular":  "delta_g_promolecular",
    "extra_feat_bond_delta_g_hirsh":         "delta_g_hirsh",
    "extra_feat_bond_esp_nuc":               "esp_nuc",
    "extra_feat_bond_esp_e":                 "esp_e",
    "extra_feat_bond_esp_total":             "esp_total",
    "extra_feat_bond_grad_norm":             "grad_norm",
    "extra_feat_bond_lap_norm":              "lap_norm",
    "extra_feat_bond_eig_hess":              "eig_hess",
    "extra_feat_bond_det_hessian":           "det_hessian",
    "extra_feat_bond_ellip_e_dens":          "ellip_e_dens",
    "extra_feat_bond_eta":                   "eta",
    "extra_feat_bond_energy_density":        "energy_density",
    "extra_feat_bond_lol":                   "lol",
}

# rebuild bond feature arrays in correct order
row_corrected = row.copy()
row_corrected["bonds"] = [correct_bonds]
row_corrected["extra_feat_bond_indices_qtaim"] = correct_bonds

for col, qkey in QTAIM_KEY_MAP.items():
    vals = [qtaim[k].get(qkey, 0.0) for k in bcp_keys]
    row_corrected[col] = vals

print(f"\n=== Corrected row bond features (first 3 bonds, Lagrangian_K) ===")
print(f"  correct bonds  : {correct_bonds[:3]}")
print(f"  Lagrangian_K   : {row_corrected['extra_feat_bond_Lagrangian_K'][:3]}")

# ── now run through the actual pipeline: mol_wrappers_from_df ─────────────────
# simulate what mol_wrappers_from_df does with the corrected row
from qtaim_embed.utils.descriptors import (
    get_atom_feats, get_bond_features, elements_from_pmg
)
from qtaim_embed.core.molwrapper import MoleculeWrapper

ATOM_KEYS = [
    "extra_feat_atom_Lagrangian_K", "extra_feat_atom_Hamiltonian_K",
    "extra_feat_atom_e_density",    "extra_feat_atom_lap_e_density",
    "extra_feat_atom_e_loc_func",
]
BOND_KEYS = BOND_FEAT_KEYS
MAP_KEY   = "extra_feat_bond_indices_qtaim"
BOND_KEY  = "bonds"

atom_feats  = get_atom_feats(row_corrected, ATOM_KEYS)
bond_feats  = get_bond_features(row_corrected, map_key=MAP_KEY,
                                 bond_key=BOND_KEY, keys=BOND_KEYS)

# unwrap bonds correctly
bonds_raw = row_corrected[BOND_KEY]
if isinstance(bonds_raw, list) and len(bonds_raw) == 1:
    bonds_raw = bonds_raw[0]
bonds_dict = {tuple(sorted(b)): None for b in bonds_raw if b[0] != b[1]}
bond_feats  = {k: v for k, v in bond_feats.items() if k[0] != k[1]}

print(f"\n=== MoleculeWrapper inputs ===")
print(f"  bonds_dict keys ({len(bonds_dict)}): {sorted(bonds_dict.keys())}")
print(f"  bond_feats keys ({len(bond_feats)}): {sorted(bond_feats.keys())}")
print(f"  atom_feats keys : {list(atom_feats.keys())}")

mol_wrapper = MoleculeWrapper(
    row_corrected["molecule_graph"],
    functional_group=None,
    free_energy=None,
    id=f"{mol_id}_{row['names']}",
    bonds=bonds_dict,
    non_metal_bonds=bonds_dict,
    atom_features=atom_feats,
    bond_features=bond_feats,
    global_features={},
    original_atom_ind=None,
    original_bond_mapping=None,
)

print(f"\n=== MoleculeWrapper built ===")
print(f"  mol_wrapper.id       = {mol_wrapper.id}")
print(f"  mol_wrapper.num_atoms= {mol_wrapper.num_atoms}")
print(f"  mol_wrapper.bonds    = {sorted(mol_wrapper.bonds.keys())}")

# ── build graph using grapher ─────────────────────────────────────────────────
import sys
sys.path.insert(0, "/home/suba/Downloads/qtaim_generator/qtaim_generator")

from qtaim_embed.data.grapher import HeteroCompleteGraphFromMolWrapper

grapher = HeteroCompleteGraphFromMolWrapper(self_loop=True)
g = grapher.build_graph(mol_wrapper)

print(f"\n=== DGL Graph built ===")
print(f"  node types : {g.ntypes}")
print(f"  edge types : {g.etypes}")
print(f"  atom nodes : {g.num_nodes('atom')}")
print(f"  bond nodes : {g.num_nodes('bond')}")
print(f"  a2b edges  : {g.num_edges('a2b')}")

# check disconnected atoms
src, dst = g.edges(etype='a2b')
atoms_with_bonds = set(src.tolist())
disconnected = [i for i in range(g.num_nodes('atom')) if i not in atoms_with_bonds]
print(f"  disconnected atoms : {disconnected} ({'NONE ✓' if not disconnected else 'PROBLEM'})")

# compare with molecule_graph
mg       = row["molecule_graph"]
mg_edges = {(min(u,v), max(u,v)) for u,v,_ in mg.graph.edges(data=True)}
g_bonds  = set(sorted(mol_wrapper.bonds.keys()))
print(f"\n=== Final check ===")
print(f"  molecule_graph edges : {len(mg_edges)}")
print(f"  graph bond nodes     : {len(g_bonds)}")
print(f"  match                : {g_bonds == mg_edges}")
print(f"  missing from graph   : {sorted(mg_edges - g_bonds)}")
print(f"  phantom in graph     : {sorted(g_bonds - mg_edges)}")

names=gdb_67265.xyz  ids=8680  gdb_num=67265

Correct bonds from ids=8680 qtaim.json (20):
  (5, 14)
  (6, 15)
  (5, 6)
  (6, 7)
  (5, 8)
  (4, 5)
  (7, 8)
  (2, 6)
  (4, 13)
  (8, 16)
  (3, 4)
  (2, 3)
  (1, 8)
  (1, 4)
  (1, 2)
  (2, 12)
  (0, 1)
  (0, 11)
  (0, 9)
  (0, 10)

=== Corrected row bond features (first 3 bonds, Lagrangian_K) ===
  correct bonds  : [(5, 14), (6, 15), (5, 6)]
  Lagrangian_K   : [0.04133085182, 0.03513779059, 0.05882509917]

=== MoleculeWrapper inputs ===
  bonds_dict keys (20): [(0, 1), (0, 9), (0, 10), (0, 11), (1, 2), (1, 4), (1, 8), (2, 3), (2, 6), (2, 12), (3, 4), (4, 5), (4, 13), (5, 6), (5, 8), (5, 14), (6, 7), (6, 15), (7, 8), (8, 16)]
  bond_feats keys (20): [(0, 1), (0, 9), (0, 10), (0, 11), (1, 2), (1, 4), (1, 8), (2, 3), (2, 6), (2, 12), (3, 4), (4, 5), (4, 13), (5, 6), (5, 8), (5, 14), (6, 7), (6, 15), (7, 8), (8, 16)]
  atom_feats keys : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]

=== MoleculeWrapper built ===
  mol_wrapper.id  

In [13]:
import os, json, numpy as np
from tqdm import tqdm

QTAIM_ROOT = "/home/suba/Downloads/qtaim_generator/qtaim_generator/data/qm9/QTAIM"

results = []
for _, row in tqdm(df_local.head(100).iterrows(), total=100):
    mol_id  = int(row["ids"])
    gdb_num = row["gdb_num"]

    qpath = os.path.join(QTAIM_ROOT, str(mol_id), "qtaim.json")
    if not os.path.exists(qpath):
        results.append((gdb_num, mol_id, "missing"))
        continue

    with open(qpath) as f:
        qtaim = json.load(f)

    # bonds from correct ids folder
    bcp_pairs = set()
    for k in qtaim:
        if '_' in str(k):
            i, j = int(k.split('_')[0]), int(k.split('_')[1])
            bcp_pairs.add((min(i,j), max(i,j)))

    # molecule_graph ground truth
    mg       = row["molecule_graph"]
    mg_edges = {(min(u,v), max(u,v)) for u,v,_ in mg.graph.edges(data=True)}

    match = bcp_pairs == mg_edges
    results.append((gdb_num, mol_id, "✓ match" if match else
                    f"✗ bcp={len(bcp_pairs)} mg={len(mg_edges)} "
                    f"miss={len(mg_edges-bcp_pairs)} phantom={len(bcp_pairs-mg_edges)}"))

n_match = sum(1 for *_, r in results if r == "✓ match")
print(f"Results for first 100 molecules using ids folder:")
print(f"  ✓ match : {n_match}/100")
print(f"  ✗ other : {100-n_match}/100")
print()
for gdb, mid, res in results:
    if res != "✓ match":
        print(f"  gdb_{gdb:6d}  ids={mid:6d}  {res}")

100%|██████████| 100/100 [00:00<00:00, 1965.73it/s]

Results for first 100 molecules using ids folder:
  ✓ match : 73/100
  ✗ other : 27/100

  gdb_ 78284  ids= 70434  ✗ bcp=23 mg=21 miss=0 phantom=2
  gdb_ 65910  ids= 55055  ✗ bcp=20 mg=19 miss=0 phantom=1
  gdb_ 96871  ids= 32719  ✗ bcp=17 mg=16 miss=0 phantom=1
  gdb_ 10217  ids= 35884  ✗ bcp=18 mg=17 miss=0 phantom=1
  gdb_ 76469  ids= 73194  ✗ bcp=22 mg=21 miss=0 phantom=1
  gdb_ 19152  ids=122018  ✗ bcp=21 mg=20 miss=0 phantom=1
  gdb_ 84747  ids=132644  ✗ bcp=19 mg=18 miss=0 phantom=1
  gdb_ 82209  ids= 59364  ✗ bcp=24 mg=23 miss=0 phantom=1
  gdb_ 50185  ids= 59536  ✗ bcp=17 mg=16 miss=0 phantom=1
  gdb_ 54000  ids=101058  ✗ bcp=20 mg=19 miss=0 phantom=1
  gdb_111996  ids= 70236  ✗ bcp=23 mg=22 miss=0 phantom=1
  gdb_106483  ids=104806  ✗ bcp=20 mg=19 miss=0 phantom=1
  gdb_ 88921  ids= 31161  ✗ bcp=17 mg=16 miss=0 phantom=1
  gdb_ 93566  ids= 46477  ✗ bcp=18 mg=17 miss=0 phantom=1
  gdb_ 89412  ids= 66489  ✗ bcp=24 mg=23 miss=0 phantom=1
  gdb_ 44802  ids=  6151  ✗ bcp=20 mg=19 

In [15]:
import os, json

QTAIM_ROOT = "/home/suba/Downloads/qtaim_generator/qtaim_generator/data/qm9/QTAIM"

# pick one phantom case to inspect
SAMPLE_GDB = 78284
row     = df_local[df_local["gdb_num"] == SAMPLE_GDB].iloc[0]
mol_id  = int(row["ids"])

with open(os.path.join(QTAIM_ROOT, str(mol_id), "qtaim.json")) as f:
    qtaim = json.load(f)

mg       = row["molecule_graph"]
mg_edges = {(min(u,v), max(u,v)) for u,v,_ in mg.graph.edges(data=True)}

atom_keys = sorted([k for k in qtaim if '_' not in str(k)], key=lambda x: int(x))
atoms     = {int(k): qtaim[k]["element"] for k in atom_keys}

bcp_keys = sorted([k for k in qtaim if '_' in str(k)],
                   key=lambda x: qtaim[x]["cp_num"])

print(f"gdb_{SAMPLE_GDB}  ids={mol_id}")
print(f"molecule_graph: {len(mg_edges)} real bonds")
print(f"qtaim.json BCPs: {len(bcp_keys)}")
print()
print(f"{'BCP':>6}  {'pair':>8}  {'sp':>6}  {'in_mg':>6}  "
      f"{'e_density':>10}  {'ellip':>8}  {'lap_e_dens':>12}")
print("-"*65)
for k in bcp_keys:
    i, j   = int(k.split('_')[0]), int(k.split('_')[1])
    pair   = (min(i,j), max(i,j))
    sp     = f"{atoms[i]}-{atoms[j]}"
    in_mg  = "✓" if pair in mg_edges else "✗ phantom"
    e_dens = qtaim[k].get("e_density", 0)
    ellip  = qtaim[k].get("ellip_e_dens", 0)
    lap    = qtaim[k].get("lap_e_density", 0)
    print(f"  {k:>4}  ({i:2d},{j:2d})  {sp:>6}  {in_mg:>8}  "
          f"{e_dens:>10.4f}  {ellip:>8.4f}  {lap:>12.4f}")

print()
print("Phantoms have low e_density and high ellipticity — non-covalent BCPs")
print("These can be filtered with a threshold on e_density or ellip_e_dens")

# find a good threshold
print()
print("e_density statistics:")
real_dens    = [abs(qtaim[k].get("e_density",0))
                for k in bcp_keys
                if (min(int(k.split('_')[0]),int(k.split('_')[1])),
                    max(int(k.split('_')[0]),int(k.split('_')[1]))) in mg_edges]
phantom_dens = [abs(qtaim[k].get("e_density",0))
                for k in bcp_keys
                if (min(int(k.split('_')[0]),int(k.split('_')[1])),
                    max(int(k.split('_')[0]),int(k.split('_')[1]))) not in mg_edges]
print(f"  real bonds    min={min(real_dens):.4f}  max={max(real_dens):.4f}")
print(f"  phantom bonds min={min(phantom_dens):.4f}  max={max(phantom_dens):.4f}")
print()
print("→ threshold e_density > X will separate real from phantom")

gdb_78284  ids=70434
molecule_graph: 21 real bonds
qtaim.json BCPs: 23

   BCP      pair      sp   in_mg   e_density     ellip    lap_e_dens
-----------------------------------------------------------------
  5_15  ( 5,15)     C-H         ✓      0.0000    0.0325       -0.9911
  6_16  ( 6,16)     C-H         ✓      0.0000    0.0609       -1.0683
   5_6  ( 5, 6)     C-C         ✓      0.0000    0.3773       -0.4653
   3_5  ( 3, 5)     C-C         ✓      0.0000    0.0423       -0.5523
  4_14  ( 4,14)     O-H         ✓      0.0000    0.0230       -2.1808
   5_8  ( 5, 8)     C-C         ✓      0.0000    0.7213       -0.2937
   6_7  ( 6, 7)     C-O         ✓      0.0000    0.0744       -0.5026
   6_8  ( 6, 8)     C-C         ✓      0.0000    0.4204       -0.4449
   3_4  ( 3, 4)     C-O         ✓      0.0000    0.1120       -0.3804
  4_17  ( 4,17)     O-H  ✗ phantom      0.0000    0.6244        0.0437
  8_18  ( 8,18)     C-H         ✓      0.0000    0.0324       -0.9987
  2_12  ( 2,12)     C-

In [16]:
pwd


'/home/suba/Documents/GitHub/qtaim_embed_private'